# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper is *The State of AI-Driven SEO, March 2026* (`docs/flyrank-seo-research-march-2026.pdf`). It is written for a broad audience, and it already sets a careful standard for itself: direct comparisons lead, ML is an exploratory appendix, and the limitations page says plainly that it's an observational study. My questions below are the ones I'd want someone to ask about my own work.

### Finding A – The refresh effect (Finding #4, "The Freshness Multiplier")
**What it reports:** pages older than 365 days that were refreshed within the last 30 days show about **3.2× the health score** and **57× the impressions** of old pages that weren't refreshed. This finding supports the #1 playbook action, "refresh mature pages before they decay".

**Where the "label" comes from:** the outcome (health score, impressions) and the grouping (days since last update) are measured in the **same 90-day snapshot**. The comparison is refreshed vs. not-refreshed pages, not the same pages before vs. after a refresh.

**My methodology questions:**
1. **How are pages chosen for a refresh?** If editors mostly refresh pages that are already doing well (or pages the workflow flags because they have impressions), the refreshed group starts out stronger. Part of the 57× could be *which* pages got picked, not what the refresh did. The paper's own Myth #3 notes that flags tend to fire on pages that already have visibility, which is the same selection mechanism.
2. **Does the timing let us read it as an effect?** Health and impressions cover a 90-day window, and "refreshed within 30 days" sits inside it. A before/after comparison on the same pages (e.g. 30 days before vs 30 days after each refresh), set against similar untouched old pages, would support the causal-sounding playbook wording much more directly.

**Constructive suggestion:** the paper's own "Measure" line already proposes refreshed vs comparable untouched pages. Making that matched before/after comparison the headline evidence would turn a strong association into a much stronger claim.

### Finding B – "What Predicts Growth?" (ML appendix)
**What it reports:** a logistic regression separates growing from declining pages with **71% holdout accuracy**, on an 80/20 split of about 61.8K active pages.

**Where the label comes from:** "Trend Direction", 30-day vs previous-30-day impressions (up = over 10% growth, down = over 10% decline).

**My methodology questions:**
1. **What is 71% compared to?** Finding #1 reports about 74.8K rising vs 45.3K falling pages, so always guessing "up" would already score about **62%** (computed below). The model's real skill is closer to ~9 points above that, which is still useful but a different headline. Printing the majority-class rate next to the accuracy would make this clear.
2. **Does the validation match the claim?** The split is a random 80/20, not grouped by brand, and some inputs (impressions, days visible) come from the 90-day window that **contains** the 30-day label window. Pages from the same brand share character, and a 90-day total already includes most of the trend being predicted. A brand-grouped split with features taken only from *before* the label window would show how much of the 71% holds up. The paper already labels this section descriptive, which is the right caution.

These are exactly the three traps I test on my own model below: base rate, grouped split, and overlapping windows.

In [1]:
# Finding B, question 1: what does always guessing the majority class score?
rising, falling = 74_187, 45_272            # cohort sizes reported in the paper's Finding #1
majority_rate = rising / (rising + falling)
print(f"Always predict 'growing': {majority_rate:.1%} accuracy")
print(f"Reported model accuracy:  71.0%  ->  about {71.0 - 100*majority_rate:.0f} points above the majority-class floor")
print("(Directional only: the appendix sample (~61.8K) is a subset, so its exact base rate may differ.)")

Always predict 'growing': 62.1% accuracy
Reported model accuracy:  71.0%  ->  about 9 points above the majority-class floor
(Directional only: the appendix sample (~61.8K) is a subset, so its exact base rate may differ.)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Setup:** exactly my Week-5 data (March 2026, one row per page, decision moment 15 March, 7 features) and my Week-5 Random Forest (300 trees, min_samples_leaf=20, seed 42). **Momentum only**, the one-feature ranking that looked so strong in Week 5, is shown next to it every time.

Three designs, from least to most honest:
1. **Before: random 5-fold split.** Pages from the same client land in both train and test.
2. **After (grouped): 5-fold GroupKFold by client.** This was my Week-5 design: every test client is unseen.
3. **After (time-aware): train on February 2026, test on March 2026.** The model is built one month and used the next, like it would be in real use. I also run a version that is time-aware *and* client-grouped (train on February's non-test clients, test on March's held-out clients).

In [2]:
import sys
sys.path.insert(0, "../scripts")
import json
import numpy as np
import pandas as pd
import lane_data as L
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold

FEATURES = ["log_imp_h1", "ctr_h1", "avg_pos_h1", "active_days_h1", "momentum_h1", "age_days", "ctr_gap"]

def prep(f):
    f = f.copy()
    f["avg_pos_h1"] = f["avg_pos_h1"].fillna(100)
    f["ctr_gap"] = f["ctr_gap"].fillna(0)
    f["momentum_h1"] = f["momentum_h1"].clip(upper=10)
    return f

def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=L.SEED, n_jobs=-1)

def evaluate(scores, labels):
    return {"P@50": L.precision_at_k(scores, labels, 50), "P@500": L.precision_at_k(scores, labels, 500),
            "ROC AUC": roc_auc_score(labels, scores), "base rate": float(np.mean(labels))}

con = L.connect()
mar = prep(L.build_frame(con, "2026-03"))
feb = prep(L.build_frame(con, "2026-02"))
print(f"March: {len(mar):,} pages, {mar['client_hash_id'].nunique()} clients, base rate {mar['is_declining'].mean():.3f}")
print(f"Feb:   {len(feb):,} pages, {feb['client_hash_id'].nunique()} clients, base rate {feb['is_declining'].mean():.3f}")

C:\Users\Itachi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


March: 92,548 pages, 40 clients, base rate 0.327
Feb:   71,333 pages, 28 clients, base rate 0.252


In [3]:
def cv_run(design, splits, frame, label="is_declining"):
    rows = []
    for a, b in splits:
        tr, te = frame.iloc[a], frame.iloc[b]
        p = rf().fit(tr[FEATURES], tr[label]).predict_proba(te[FEATURES])[:, 1]
        rows.append({"design": design, "method": "Random Forest", **evaluate(p, te[label])})
        rows.append({"design": design, "method": "Momentum only", **evaluate(-te["momentum_h1"], te[label])})
    return rows

rows = []
rows += cv_run("1. BEFORE: random 5-fold", KFold(5, shuffle=True, random_state=L.SEED).split(mar), mar)
rows += cv_run("2. AFTER: grouped by client (5-fold)", GroupKFold(5).split(mar, groups=mar["client_hash_id"]), mar)

# Time-aware: fit on February, score March
p = rf().fit(feb[FEATURES], feb["is_declining"]).predict_proba(mar[FEATURES])[:, 1]
rows.append({"design": "3. AFTER: time-aware (Feb -> Mar)", "method": "Random Forest", **evaluate(p, mar["is_declining"])})
rows.append({"design": "3. AFTER: time-aware (Feb -> Mar)", "method": "Momentum only", **evaluate(-mar["momentum_h1"], mar["is_declining"])})

# Time-aware AND client-grouped: Feb pages of non-test clients -> March pages of the Week-5 test clients
_, mar_test = L.client_split(mar)
feb_train = feb[~feb["client_hash_id"].isin(mar_test["client_hash_id"])]
p = rf().fit(feb_train[FEATURES], feb_train["is_declining"]).predict_proba(mar_test[FEATURES])[:, 1]
rows.append({"design": "4. AFTER: time + client (Feb -> Mar, unseen clients)", "method": "Random Forest", **evaluate(p, mar_test["is_declining"])})
rows.append({"design": "4. AFTER: time + client (Feb -> Mar, unseen clients)", "method": "Momentum only", **evaluate(-mar_test["momentum_h1"], mar_test["is_declining"])})

before_after = pd.DataFrame(rows).groupby(["design", "method"], sort=True).mean().round(3)
before_after

P@50  \
design                                             method                 
1. BEFORE: random 5-fold                           Momentum only  0.928   
                                                   Random Forest  0.944   
2. AFTER: grouped by client (5-fold)               Momentum only  0.932   
                                                   Random Forest  0.884   
3. AFTER: time-aware (Feb -> Mar)                  Momentum only  0.940   
                                                   Random Forest  0.780   
4. AFTER: time + client (Feb -> Mar, unseen cli... Momentum only  0.940   
                                                   Random Forest  0.960   

                                                                  P@500  \
design                                             method                 
1. BEFORE: random 5-fold                           Momentum only  0.795   
                                                   Random Forest  0.871   
2. AFTER: grouped by client (5-fold)               Momentum only  0.790   
                                                   Random Forest  0.789   
3. AFTER: time-aware (Feb -> Mar)                  Momentum only  0.912   
                                                   Random Forest  0.812   
4. AFTER: time + client (Feb -> Mar, unseen cli... Momentum only  0.898   
                                                   Random Forest  0.840   

                                                                  ROC AUC  \
design                                             method                   
1. BEFORE: random 5-fold                           Momentum only    0.674   
                                                   Random Forest    0.762   
2. AFTER: grouped by client (5-fold)               Momentum only    0.669   
                                                   Random Forest    0.690   
3. AFTER: time-aware (Feb -> Mar)                  Momentum only    0.674   
                                                   Random Forest    0.650   
4. AFTER: time + client (Feb -> Mar, unseen cli... Momentum only    0.756   
                                                   Random Forest    0.703   

                                                                  base rate  
design                                             method                    
1. BEFORE: random 5-fold                           Momentum only      0.327  
                                                   Random Forest      0.327  
2. AFTER: grouped by client (5-fold)               Momentum only      0.327  
                                                   Random Forest      0.327  
3. AFTER: time-aware (Feb -> Mar)                  Momentum only      0.327  
                                                   Random Forest      0.327  
4. AFTER: time + client (Feb -> Mar, unseen cli... Momentum only      0.405  
                                                   Random Forest      0.405

**What the before/after shows:**
- **The random split flattered the Random Forest.** Going from random to client-grouped, RF drops from about **0.94 → 0.88** P@50 and **0.76 → 0.69** ROC AUC. That gap is memorization: in a random split the model learns each client's habits from its other pages. Momentum-only doesn't change, which makes sense because it learns nothing.
- **Across time it's worse.** Trained on February and tested on March, RF's AUC falls to about **0.65**, *below* the no-model momentum ranking (~0.67). Whatever extra the forest learned in one month doesn't carry over cleanly to the next. February also has a different base rate (0.25 vs 0.33).
- **Under the honest designs, the Random Forest's edge over the no-model momentum ranking is small and inconsistent.** It wins on ROC AUC in the grouped split (0.690 vs 0.669), but loses on AUC in both time-aware designs (0.650 vs 0.674, and 0.703 vs 0.756 on unseen clients), even though its top-50 there is slightly higher (0.96 vs 0.94). A one-feature ranking that holds up this well made me suspicious of the label itself, and that's what section 3 checks.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The checklist from the leakage skill, run on my final Week-5 feature set:

| Check | How I test it | Result |
|---|---|---|
| Timeline: all features before the label window | features filter `report_date <= 15th`, label `> 15th` (code in `lane_data.py`); assert below | pass |
| No label-derived columns | deliberate-leak test with `imp_h2` (should jump toward 1.0) | pass: harness works, `imp_h2` is not a feature |
| No product flags / future metadata | `content_updated_date`, `last_optimized_date` excluded (mostly dated after the decision) | pass |
| Nothing "too good" | single-feature AUC scan | pass: best single feature ≈ 0.67 |
| **Label mechanics** | does the label partly *contain* a feature? | **FAIL, see below** |

In [4]:
aud = prep(L.build_frame(con, "2026-03", keep_future=True))   # imp_h2 kept ONLY for this audit

# 1) Timeline: the SQL windows (from lane_data) - features <= 15th, label > 15th
import inspect
src = inspect.getsource(L.build_frame)
assert "report_date <= DATE '{decision_date}'" in src and "report_date >  DATE '{decision_date}') AS imp_h2" in src
print("1) Timeline check passed: features use days 1-15, only imp_h2 (the label source) uses days 16+")

# 2) Deliberate leak: add imp_h2 and watch the grouped score jump
g = list(GroupKFold(5).split(aud, groups=aud["client_hash_id"]))[0]
tr, te = aud.iloc[g[0]], aud.iloc[g[1]]
honest = roc_auc_score(te["is_declining"], rf().fit(tr[FEATURES], tr["is_declining"]).predict_proba(te[FEATURES])[:, 1])
leaky = roc_auc_score(te["is_declining"], rf().fit(tr[FEATURES + ["imp_h2"]], tr["is_declining"]).predict_proba(te[FEATURES + ["imp_h2"]])[:, 1])
print(f"2) Deliberate leak (fold 1): honest AUC {honest:.3f} -> with imp_h2 {leaky:.3f}  (harness catches leaks)")

# 3) Single-feature AUC scan: anything suspiciously perfect?
scan = pd.Series({c: max(roc_auc_score(aud["is_declining"], aud[c]), 1 - roc_auc_score(aud["is_declining"], aud[c])) for c in FEATURES})
print("3) Single-feature AUC (direction-free):")
print(scan.sort_values(ascending=False).round(3).to_string())

1) Timeline check passed: features use days 1-15, only imp_h2 (the label source) uses days 16+


2) Deliberate leak (fold 1): honest AUC 0.661 -> with imp_h2 0.976  (harness catches leaks)


3) Single-feature AUC (direction-free):
momentum_h1       0.674
ctr_gap           0.574
ctr_h1            0.565
active_days_h1    0.561
age_days          0.535
avg_pos_h1        0.517
log_imp_h1        0.503


### The finding: my label partly contains my top feature

My label compares days 16–31 with the **average of days 1–15**. `momentum_h1` compares week 2 with week 1 *inside* those same days 1–15. If a page dropped in week 2 and then just **stays flat** at that lower level, its 16–31 average is automatically below its 1–15 average, so it gets labelled "declining" without losing anything further. Part of "momentum predicts decline" is really "the label counts a past drop as a future decline". It isn't future leakage (everything is known on the 15th), but it is a **label-construction artifact**. It's the same kind of question I asked about the paper's trend label.

**Test:** a stricter label that only counts **further** decline, meaning days 16–31 below 80% of the page's **latest week** (8–15). If momentum were real foresight, it should still work.

In [5]:
aud["is_declining_strict"] = ((aud["imp_h2"] / aud["h2_days"]) < 0.8 * (aud["imp_wk2"] / 8)).astype(int)

mech = pd.DataFrame({
    "label": ["original (vs days 1-15 average)", "strict (vs latest week 8-15)"],
    "base rate": [aud["is_declining"].mean(), aud["is_declining_strict"].mean()],
    "momentum-only AUC": [roc_auc_score(aud["is_declining"], -aud["momentum_h1"]),
                          roc_auc_score(aud["is_declining_strict"], -aud["momentum_h1"])],
    "momentum-only P@50": [L.precision_at_k(-aud["momentum_h1"], aud["is_declining"], 50),
                           L.precision_at_k(-aud["momentum_h1"], aud["is_declining_strict"], 50)],
}).round(3)
print(mech.to_string(index=False))

print("\nDecline rate by momentum bucket, under each label:")
print(aud.groupby(pd.cut(aud["momentum_h1"], [-0.01, 0.5, 0.9, 1.1, 2, 10],
                         labels=["<0.5 falling", "0.5-0.9", "0.9-1.1 flat", "1.1-2", ">2 spiking"]), observed=True)
         [["is_declining", "is_declining_strict"]].mean().round(3).to_string())

                          label  base rate  momentum-only AUC  momentum-only P@50
original (vs days 1-15 average)      0.327              0.674                0.94
   strict (vs latest week 8-15)      0.301              0.490                0.00

Decline rate by momentum bucket, under each label:
              is_declining  is_declining_strict
momentum_h1                                    
<0.5 falling         0.655                0.335
0.5-0.9              0.379                0.275
0.9-1.1 flat         0.275                0.289
1.1-2                0.240                0.342
>2 spiking           0.129                0.294


Under the strict label **momentum's skill disappears completely** (AUC ≈ 0.49, which is chance). The falling pages that looked 66% "declining" only decline *further* 34% of the time, about the base rate. So most of my Week-5 headline came from how the label was built.

**Does anything real survive?** I re-run the grouped comparison on the strict label: Random Forest vs momentum-only vs my Week-4 baseline rule. I only use pages with at least 25 impressions in week 2, so "latest level" means something.

In [6]:
strict = aud[aud["imp_wk2"] >= 25].reset_index(drop=True)
strict_scored = L.baseline_score(strict)

rows, oof = [], np.zeros(len(strict))
for a, b in GroupKFold(5).split(strict, groups=strict["client_hash_id"]):
    tr, te = strict.iloc[a], strict.iloc[b]
    p = rf().fit(tr[FEATURES], tr["is_declining_strict"]).predict_proba(te[FEATURES])[:, 1]
    oof[b] = p
    y = te["is_declining_strict"]
    rows += [{"method": "Random Forest", **evaluate(p, y)},
             {"method": "Momentum only", **evaluate(-te["momentum_h1"], y)},
             {"method": "Week-4 baseline rule", **evaluate(strict_scored.iloc[b]["baseline_score"], y)}]
strict_table = pd.DataFrame(rows).groupby("method").agg(["mean", "std"]).round(3)
print(f"Strict label, pages with >=25 week-2 impressions: n={len(strict):,}, base rate {strict['is_declining_strict'].mean():.3f}")
strict_table

Strict label, pages with >=25 week-2 impressions: n=90,144, base rate 0.301


P@50         P@500        ROC AUC        base rate  \
                       mean    std   mean    std    mean    std      mean   
method                                                                      
Momentum only         0.424  0.192  0.334  0.078   0.478  0.062     0.301   
Random Forest         0.604  0.158  0.532  0.133   0.608  0.024     0.301   
Week-4 baseline rule  0.348  0.090  0.317  0.037   0.560  0.026     0.301   

                             
                        std  
method                       
Momentum only         0.071  
Random Forest         0.071  
Week-4 baseline rule  0.071

**Honest result on the strict label (5-fold, grouped by client):** the Random Forest still beats both simple rankings, with P@50 about **0.60** against a base rate of about **0.30**, and ROC AUC about **0.61**. Momentum-only is at chance on AUC (≈0.48), and the Week-4 rule is slightly above chance (AUC ≈0.56, P@50 ≈0.35). So there **is** a real, measured signal, but it's modest, and much smaller than the 0.88–0.93 I reported in Week 5. P@50 also swings a lot between folds (std ≈0.16), so it's directional.

### Failure examples (strict label, out-of-fold predictions)

In [7]:
strict["rf_oof"] = oof
show = ["content_hash_id", "rf_oof", "is_declining_strict", "imp_h1", "imp_wk1", "imp_wk2", "avg_pos_h1", "active_days_h1", "age_days"]
fa = strict[strict["is_declining_strict"] == 0].nlargest(3, "rf_oof").assign(error="false alarm")
md = strict[(strict["is_declining_strict"] == 1) & (strict["imp_wk2"] >= 500)].nsmallest(2, "rf_oof").assign(error="missed decline")
examples = pd.concat([fa, md])
examples["content_hash_id"] = "…" + examples["content_hash_id"].str[-6:]
examples[["error"] + show].round(3)

,error,content_hash_id,rf_oof,is_declining_strict,imp_h1,imp_wk1,imp_wk2,avg_pos_h1,active_days_h1,age_days
86295,false alarm,…0821be,0.870,0,4821.0,2012.0,2809.0,34.481,15,137
74684,false alarm,…9d678f,0.864,0,6147.0,2780.0,3367.0,44.513,13,144
71402,false alarm,…6a11e4,0.861,0,6776.0,2897.0,3879.0,37.979,13,144
43190,missed decline,…531178,0.026,1,3262.0,1667.0,1595.0,2.690,15,249
64237,missed decline,…677ffb,0.031,1,6363.0,3612.0,2751.0,2.303,15,244


**What these failures have in common:**
- **False alarms:** all three are mid-size pages (about 5–7K impressions) sitting **deep in the results (average position 34–45)** whose impressions **rose** in week 2. The model seems to have learned that a rise at a deep position usually fades back, but these pages held their higher week-2 level. It's a reasonable pattern that fails for individual pages.
- **Missed declines:** steady **top-3 pages** (average position 2.3–2.7, visible all 15 days, about 8 months old) with flat or slightly softening week 2, which then dropped in the second half. Nothing in the 1–15 March signals warned of it. These are probably external events (a competitor, an algorithm change, a seasonal query) that my seven features can't see at all.
- **What would help:** query-level signals (`fact_content_query_90d`, restricted to the pre-decision window) and a longer history per page, so the model can tell "normal wobble" from "real slide".

**Rechecking the product-flag and future-metadata trap on the final set:**

In [8]:
dates = con.sql(f"""
    SELECT ROUND(AVG((content_updated_date > DATE '{L.DECISION_DATE}')::INT), 3) AS updated_after_decision,
           ROUND(AVG((last_optimized_date  > DATE '{L.DECISION_DATE}')::INT), 3) AS optimized_after_decision,
           ROUND(AVG((content_created_date > DATE '{L.DECISION_DATE}')::INT), 3) AS created_after_decision
    FROM {L.DIM_CONTENT}
    WHERE content_hash_id IN (SELECT content_hash_id FROM mar)
""").df()
print("Share of dim_content dates falling AFTER the decision moment (only created_date is used, as age):")
print(dates.to_string(index=False))
print("Features used:", FEATURES)
print("Any flag / update-date column in features?", any(("flag" in c) or ("updated" in c) or ("optimized" in c) for c in FEATURES))

with open("../outputs/w06_audit_metrics.json", "w") as fh:
    json.dump({"before_after": before_after.reset_index().to_dict(orient="records"),
               "label_mechanics": mech.to_dict(orient="records"),
               "strict_label_grouped_5fold": strict_table.set_axis([f"{m} {s}" for m, s in strict_table.columns], axis=1)
                                                         .reset_index().to_dict(orient="records")}, fh, indent=2, default=str)
print("Saved work/outputs/w06_audit_metrics.json")

Share of dim_content dates falling AFTER the decision moment (only created_date is used, as age):
 updated_after_decision  optimized_after_decision  created_after_decision
                  0.808                       1.0                     0.0
Features used: ['log_imp_h1', 'ctr_h1', 'avg_pos_h1', 'active_days_h1', 'momentum_h1', 'age_days', 'ctr_gap']
Any flag / update-date column in features? False
Saved work/outputs/w06_audit_metrics.json


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentences from earlier weeks, rewritten to match the evidence I now have:

| Week | What I wrote | Why it goes too far | Rewrite (safe language) |
|---|---|---|---|
| 5 | "on clients it has never seen, the top 50 by momentum is right about 93% of the time" | Most of that precision comes from the label counting a week-2 drop that had **already happened**. Under a label that only counts further decline, momentum is at chance | "On March 2026 data, ranking by week-2 momentum **matched** the original decline label at P@50 ≈ 0.93, but that label partly **re-measures a drop that had already happened**. Under a stricter label (further decline from the latest week) momentum showed **no measured skill**." |
| 5 | "Every learned model beats the Week-4 rule by a wide margin" | True on the original label, but the margin was inflated by the same artifact | "On the original label every model **measured** well above my rule. On the stricter label the Random Forest's lead is **modest but still present** (P@50 ≈ 0.60 vs ≈ 0.35 for the rule, base rate ≈ 0.30)." |
| 5 | "For scoring the full list, I keep the Random Forest" | Its extra AUC over momentum disappeared when tested on a later month | "The Random Forest is my **decision-support** candidate for the strict label. Across months (Feb → Mar) its advantage **did not carry over** on the original label, so it needs a time-aware check on the strict label before any real use." |
| 3 | "honest score ROC AUC = 0.754, Precision@50 = 0.96" | A single random-seed client split on the artifact-prone label | "A **single** client-grouped split gave AUC ≈ 0.75. Grouped 5-fold gives ≈ 0.69, and the stricter label ≈ 0.61. I report the lowest honest number as my headline." |

**My new headline claim:** *"On one month (March 2026) of anonymized FlyRank data, a Random Forest built only from first-half-of-month search signals **ranked pages that went on to decline further** better than chance and better than my hand-written rule (grouped 5-fold P@50 ≈ 0.60 vs base rate ≈ 0.30; ROC AUC ≈ 0.61). This is an **observed, directional** result for **decision-support** only: one month, one label definition, no causal claim about why pages decline."*

**What changes going forward:** the strict label becomes my main label, the models get re-compared on it with the time-aware check, and every metric gets printed next to its base rate.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.